# 03 — Marcadores lingüísticos
**Autor:** Giuliano Crenna, Juan Ignacio Pace (UGR)
**Fecha:** 2026-09-03
**Descripción:** Frecuencia de 1ra persona singular, vocabulario absolutista,
negatividad — por clase. (Hipótesis H1 de la tesis.)
## Parámetros
- `DATA_DIR`, `SEED`, `OUT_DIR` (papermill).


In [0]:
# %% [code]
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve().parent))  # para que `import src.*` funcione
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
SEED = int(os.environ.get("SEED", 42))
OUT_DIR = Path(os.environ.get("OUT_DIR", "reports"))
np.random.seed(SEED)
from src.features.liwc_counts import count_markers
from src.features.polarity import score
df = pd.read_parquet(DATA_DIR / "processed" / "corpus_v1.parquet")
print(f"corpus: {len(df)} filas")


In [0]:
# %% [code]
# Calcular marcadores LIWC.
lex_path = Path("src/features/lexicons/leis_lexicon.csv")
markers = count_markers(df["text_clean"].fillna("").tolist(), lexicon_path=lex_path)
df = pd.concat([df.reset_index(drop=True), markers], axis=1)
print(markers.describe())


In [0]:
# %% [code]
# Promedio de marcadores por clase.
cat_cols = [c for c in markers.columns if c.endswith("_norm")]
agg = df.groupby("label")[cat_cols].mean()
print(agg)


In [0]:
# %% [code]
# Heatmap de marcadores normalizados.
fig, ax = plt.subplots(figsize=(9, 3))
sns.heatmap(agg.T, annot=True, fmt=".4f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Marcadores normalizados por clase")
plt.tight_layout()
out = OUT_DIR / "figures" / "eda_03_marcadores_por_clase.png"
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=120)
plt.show()


In [0]:
# %% [code]
# Polaridad.
pol = score(df["text_clean"].fillna("").tolist())
df = pd.concat([df.reset_index(drop=True), pol], axis=1)
fig, ax = plt.subplots(figsize=(7, 4))
df.boxplot(column="polarity", by="label", ax=ax)
ax.set_title("Polaridad por clase")
ax.set_xlabel("label (0=control, 2=depresivo)")
plt.suptitle("")
plt.tight_layout()
out = OUT_DIR / "figures" / "eda_03_polaridad_por_clase.png"
plt.savefig(out, dpi=120)
plt.show()


## Conclusiones esperadas (H1)
- 1ra persona singular ↑ en clase depresiva (Leis 2019 ya lo mostró).
- Absolutistas ↑ en clase depresiva.
- Polaridad (compound) ↓ en clase depresiva.
